# 08 - Scala basics (via the Almond kernel)

This notebook demonstrates that the **Scala kernel** is working in JupyterHub and shows the patterns you'll use to interact with the stack from Scala cells.

## Prerequisites

Open this notebook with the **Scala 3** kernel (not Python). Pick it from:
- **JupyterLab launcher**: click the `Scala 3` tile, then open this file.
- **Kernel picker** (top-right): `Change kernel...` → `Scala 3`.
- **VS Code**: `Select kernel` → `Scala 3` on the JupyterHub server.

For Scala 2.13, the notebook ships a parallel `scala213` kernel — open this file with that picker and the syntax in cells §1–§3 still works. The only Scala-3-specific cell is §4 (enums + extension methods); it will not parse under 2.13.

## What's installed

Both kernels are baked into the JupyterHub image via `services/jupyterhub/build/Dockerfile` (Almond `0.14.5`, Scala `2.13.16` + `3.4.3`, OpenJDK 17). If you don't see them in the kernel picker, the image was built before the kernels were added — rebuild with `docker compose up jupyterhub --build --no-deps -d` and reload the JupyterLab tab.

## 1. Basic Scala syntax

Variables, values, functions, lambdas — confirms the kernel evaluates code.

In [ ]:
// `val` is immutable, `var` is mutable. Prefer `val`.
val greeting = "Hello from Scala"
var counter = 0
counter += 1

// Functions: `def name(arg: Type): ReturnType = body`. Return inferred when omitted.
def square(x: Int) = x * x

// Lambdas + collection ops.
val squares = (1 to 5).map(square).toList

println(s"$greeting (call #$counter): $squares")

## 2. Pulling in a library at runtime

Almond's `import $ivy` (a coursier alias for SBT `libraryDependencies`) lets you add Maven artifacts directly inside a cell — no `build.sbt` required. The first run downloads + caches under `~/.cache/coursier/`.

In [ ]:
import $ivy.`com.lihaoyi::upickle:3.3.1`
import upickle.default.*

// upickle is a fast, dependency-free JSON encoder/decoder.
case class Voice(name: String, language: String) derives ReadWriter

val voices = List(Voice("kokoro", "en"), Voice("piper", "en-US"))
val json = write(voices, indent = 2)
println(json)

// Roundtrip.
val parsed = read[List[Voice]](json)
println(s"Roundtrip: ${parsed.head}")

## 3. Calling the LiteLLM gateway

The container has `LITELLM_BASE_URL` (e.g. `http://litellm:4000`) and `LITELLM_API_KEY` pre-populated in the environment — same shape Python notebooks use. Scala can hit them via the JDK's built-in `java.net.http.HttpClient`, no extra dependency.

In [ ]:
import java.net.URI
import java.net.http.{HttpClient, HttpRequest, HttpResponse}
import upickle.default.*

case class Choice(message: Message)
case class Message(role: String, content: String) derives ReadWriter
case class ChatResponse(choices: List[Choice]) derives ReadWriter
object Choice { given ReadWriter[Choice] = macroRW }

val base = sys.env.getOrElse("LITELLM_BASE_URL", "http://litellm:4000")
val key  = sys.env.getOrElse("LITELLM_API_KEY",  "<not-set>")

val body = write(ujson.Obj(
  "model"    -> "ollama/qwen3.6:latest",
  "messages" -> ujson.Arr(ujson.Obj("role" -> "user", "content" -> "Reply with just the word 'pong'."))
))

val req = HttpRequest.newBuilder()
  .uri(URI.create(s"$base/v1/chat/completions"))
  .header("Authorization", s"Bearer $key")
  .header("Content-Type",  "application/json")
  .POST(HttpRequest.BodyPublishers.ofString(body))
  .build()

val client = HttpClient.newHttpClient()
val resp   = client.send(req, HttpResponse.BodyHandlers.ofString())

println(s"HTTP ${resp.statusCode}")
if resp.statusCode == 200 then
  val parsed = read[ChatResponse](resp.body)
  println(s"Model says: ${parsed.choices.head.message.content}")
else
  println(s"Body: ${resp.body.take(200)}")

## 4. Scala 3 features

A few things Scala 3 unlocks over 2.13. Open this notebook with the **Scala 2.13** kernel and this cell will fail to compile — that's expected.

In [ ]:
// Enums (no more sealed-trait + case-object boilerplate).
enum TtsEngine:
  case Kokoro, Piper, Chatterbox
  def isCloningCapable: Boolean = this == Chatterbox

// Extension methods (no more implicit-class boilerplate).
extension (s: String)
  def snakeCase: String = s.replaceAll("([a-z])([A-Z])", "$1_$2").toLowerCase

println(TtsEngine.values.map(e => s"${e.toString.snakeCase} -> cloning=${e.isCloningCapable}").mkString("\n"))

## 5. Where to go next

- **Almond documentation:** [almond.sh/docs](https://almond.sh/docs/intro)
- **Coursier** (the dependency-resolver behind `$ivy`): [get-coursier.io](https://get-coursier.io/)
- **Connecting from VS Code**: see `services/jupyterhub/README.md` §10. The Scala kernels work identically over the remote-Jupyter wire as Python — VS Code's `Select kernel` picker lists `Scala 2.13` and `Scala 3` alongside `Python 3 (ipykernel)`.
- **Spark from a notebook**: see `10_spark_scala.ipynb` — Spark from Scala via the Spark Connect client (`org.apache.spark::spark-connect-client-jvm:4.1.2`) talking to the in-stack `spark-connect` sidecar. The Python equivalent is `09_spark_connect.ipynb`. (Ray remains the Python-native distributed path — see `07_ray_cluster.ipynb`.)